# SRLG Colocation Derivation Pipeline

**Project:** Provable Resilience Intelligence — Lakelink Fiber  
**Purpose:** Derive Shared Risk Link Groups (SRLGs) from physical topology colocation analysis.  
**Output:** `cdm_tmforum.provable_resilience_intelligence.derived_shared_risk_group`  
**Source:** physical_resource_span, physical_resource_conduit, geographic_site  

## Derivation Strategies
1. **Shared Conduit** — Spans sharing the same conduit_id are fate-sharing (fiber cut affects all)
2. **Common Site** — Spans terminating at the same central office or data center (CO/DC failure affects all)
3. **Geographic Proximity** — Span endpoints within 100m of each other (bridge, tunnel, ROW colocation)

Severity assignment:
- shared_conduit → **critical** (direct physical fate-sharing)
- central_office → **high** (site failure cascades)
- geographic_proximity → **medium** (potential colocation risk)

In [0]:
%sql
-- Strategy 1: Shared Conduit — spans in the same conduit are fate-sharing
-- Each conduit_id with 2+ spans becomes one SRLG
CREATE OR REPLACE TABLE cdm_tmforum.provable_resilience_intelligence.derived_srlg_shared_conduit
COMMENT 'Lakelink Fiber — Derived SRLGs from shared conduit colocation. Spans sharing a conduit_id are physically fate-sharing: a single fiber cut or conduit breach affects all member spans.'
AS
WITH conduit_groups AS (
  SELECT 
    conduit_id,
    COLLECT_LIST(span_id) AS member_spans,
    COUNT(*) AS member_count
  FROM cdm_tmforum.provable_resilience_intelligence.physical_resource_span
  WHERE conduit_id IS NOT NULL AND is_active = TRUE
  GROUP BY conduit_id
  HAVING COUNT(*) >= 2
)
SELECT
  CONCAT('SRLG-COND-', conduit_id) AS srlg_id,
  CONCAT('Shared Conduit ', conduit_id) AS srlg_name,
  'shared_conduit' AS srlg_type,
  span_id,
  conduit_id,
  CAST(NULL AS STRING) AS site_id,
  -- Risk point = midpoint of conduit (avg of member span endpoints)
  AVG(s.latitude) OVER (PARTITION BY cg.conduit_id) AS risk_latitude,
  AVG(s.longitude) OVER (PARTITION BY cg.conduit_id) AS risk_longitude,
  'critical' AS severity,
  member_count,
  CURRENT_TIMESTAMP() AS derived_at
FROM conduit_groups cg
LATERAL VIEW EXPLODE(member_spans) AS span_id
JOIN cdm_tmforum.provable_resilience_intelligence.physical_resource_span sp
  ON span_id = sp.span_id
JOIN cdm_tmforum.provable_resilience_intelligence.geographic_site s
  ON sp.site_a_id = s.site_id

In [0]:
%sql
-- Strategy 2: Common Site — spans terminating at the same site share CO/DC failure risk
-- Each site with 2+ span terminations becomes one SRLG
CREATE OR REPLACE TABLE cdm_tmforum.provable_resilience_intelligence.derived_srlg_common_site
COMMENT 'Lakelink Fiber — Derived SRLGs from common site colocation. Spans sharing a termination site (central office, data center) are at risk of simultaneous failure from site-level events (power, cooling, fire).'
AS
WITH site_spans AS (
  -- Collect all spans that terminate at each site (either A or Z end)
  SELECT site_id, COLLECT_SET(span_id) AS member_spans
  FROM (
    SELECT site_a_id AS site_id, span_id 
    FROM cdm_tmforum.provable_resilience_intelligence.physical_resource_span WHERE is_active = TRUE
    UNION
    SELECT site_z_id AS site_id, span_id 
    FROM cdm_tmforum.provable_resilience_intelligence.physical_resource_span WHERE is_active = TRUE
  )
  WHERE site_id IS NOT NULL
  GROUP BY site_id
  HAVING SIZE(COLLECT_SET(span_id)) >= 2
)
SELECT
  CONCAT('SRLG-SITE-', ss.site_id) AS srlg_id,
  CONCAT('Site Colocation ', gs.site_name) AS srlg_name,
  'central_office' AS srlg_type,
  span_id,
  CAST(NULL AS STRING) AS conduit_id,
  ss.site_id,
  gs.latitude AS risk_latitude,
  gs.longitude AS risk_longitude,
  'high' AS severity,
  SIZE(member_spans) AS member_count,
  CURRENT_TIMESTAMP() AS derived_at
FROM site_spans ss
LATERAL VIEW EXPLODE(member_spans) AS span_id
JOIN cdm_tmforum.provable_resilience_intelligence.geographic_site gs
  ON ss.site_id = gs.site_id

In [0]:
%sql
-- Strategy 3: Geographic Proximity — span endpoints within 100m share ROW/bridge/tunnel risk
-- Uses Haversine formula to find site pairs within 100m, then collects their spans
CREATE OR REPLACE TABLE cdm_tmforum.provable_resilience_intelligence.derived_srlg_geo_proximity
COMMENT 'Lakelink Fiber — Derived SRLGs from geographic proximity colocation. Sites within 100m of each other share right-of-way, bridge, or tunnel risk. Their spans may be affected by the same physical event (dig, flood, collapse).'
AS
WITH site_pairs AS (
  -- Find site pairs within 100m using Haversine bounding box + exact distance
  SELECT 
    s1.site_id AS site_id_1,
    s2.site_id AS site_id_2,
    -- Haversine distance in meters
    6371000 * 2 * ASIN(SQRT(
      POWER(SIN(RADIANS(s2.latitude - s1.latitude) / 2), 2) +
      COS(RADIANS(s1.latitude)) * COS(RADIANS(s2.latitude)) *
      POWER(SIN(RADIANS(s2.longitude - s1.longitude) / 2), 2)
    )) AS distance_m,
    (s1.latitude + s2.latitude) / 2 AS midpoint_lat,
    (s1.longitude + s2.longitude) / 2 AS midpoint_lon
  FROM cdm_tmforum.provable_resilience_intelligence.geographic_site s1
  JOIN cdm_tmforum.provable_resilience_intelligence.geographic_site s2
    ON s1.site_id < s2.site_id  -- avoid duplicates
    -- Bounding box pre-filter (~0.001 degrees ≈ 111m at equator)
    AND ABS(s1.latitude - s2.latitude) < 0.001
    AND ABS(s1.longitude - s2.longitude) < 0.001
  WHERE s1.is_active = TRUE AND s2.is_active = TRUE
),
close_pairs AS (
  SELECT * FROM site_pairs WHERE distance_m <= 100
),
proximity_spans AS (
  -- Collect spans from both sites in each close pair
  SELECT
    cp.site_id_1,
    cp.site_id_2,
    cp.distance_m,
    cp.midpoint_lat,
    cp.midpoint_lon,
    COLLECT_SET(sp.span_id) AS member_spans
  FROM close_pairs cp
  JOIN cdm_tmforum.provable_resilience_intelligence.physical_resource_span sp
    ON (sp.site_a_id IN (cp.site_id_1, cp.site_id_2)
     OR sp.site_z_id IN (cp.site_id_1, cp.site_id_2))
    AND sp.is_active = TRUE
  GROUP BY cp.site_id_1, cp.site_id_2, cp.distance_m, cp.midpoint_lat, cp.midpoint_lon
  HAVING SIZE(COLLECT_SET(sp.span_id)) >= 2
)
SELECT
  CONCAT('SRLG-GEO-', site_id_1, '-', site_id_2) AS srlg_id,
  CONCAT('Proximity ', site_id_1, ' / ', site_id_2, ' (', ROUND(distance_m, 1), 'm)') AS srlg_name,
  'geographic_proximity' AS srlg_type,
  span_id,
  CAST(NULL AS STRING) AS conduit_id,
  site_id_1 AS site_id,
  midpoint_lat AS risk_latitude,
  midpoint_lon AS risk_longitude,
  'medium' AS severity,
  SIZE(member_spans) AS member_count,
  CURRENT_TIMESTAMP() AS derived_at
FROM proximity_spans
LATERAL VIEW EXPLODE(member_spans) AS span_id

In [0]:
%sql
-- Union all 3 derivation strategies into unified derived SRLG table
CREATE OR REPLACE TABLE cdm_tmforum.provable_resilience_intelligence.derived_shared_risk_group
COMMENT 'Lakelink Fiber — Unified derived Shared Risk Link Groups (SRLGs) from 3 colocation strategies: shared conduit (critical), common site (high), and geographic proximity within 100m (medium). Production replacement for synthetic shared_risk_group.'
AS
SELECT srlg_id, srlg_name, srlg_type, span_id, conduit_id, site_id, 
       risk_latitude, risk_longitude, severity, member_count, derived_at
FROM cdm_tmforum.provable_resilience_intelligence.derived_srlg_shared_conduit
UNION ALL
SELECT srlg_id, srlg_name, srlg_type, span_id, conduit_id, site_id, 
       risk_latitude, risk_longitude, severity, member_count, derived_at
FROM cdm_tmforum.provable_resilience_intelligence.derived_srlg_common_site
UNION ALL
SELECT srlg_id, srlg_name, srlg_type, span_id, conduit_id, site_id, 
       risk_latitude, risk_longitude, severity, member_count, derived_at
FROM cdm_tmforum.provable_resilience_intelligence.derived_srlg_geo_proximity

In [0]:
%sql
-- Validation: profile derived SRLGs
SELECT 
  srlg_type,
  severity,
  COUNT(*) AS total_memberships,
  COUNT(DISTINCT srlg_id) AS distinct_srlgs,
  COUNT(DISTINCT span_id) AS distinct_spans,
  ROUND(AVG(member_count), 1) AS avg_members_per_srlg,
  MAX(member_count) AS max_members
FROM cdm_tmforum.provable_resilience_intelligence.derived_shared_risk_group
GROUP BY srlg_type, severity
ORDER BY 
  CASE severity WHEN 'critical' THEN 1 WHEN 'high' THEN 2 WHEN 'medium' THEN 3 ELSE 4 END